In [2]:
import numpy as np
import scipy.constants
import scipy.sparse
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import math
import geopandas as gpd
import pandas as pd
import os
import netCDF4 as nc

In [3]:
def calc_alongshore_transport_k(gravity=scipy.constants.g, n=1.0, rho_water=1050.0, gamma_b=0.78, ):

    '''Calculates sediment transport constant k.'''
    
    return (
        5.3e-6
        # * 0.46
        * rho_water
        * gravity**1.5
        * (1 / (2 * n)) ** 1.2
        * (np.sqrt(gravity * gamma_b) / (2 * np.pi)) ** 0.2
    )

In [4]:
def get_angles_xr(ds):
    
    ''' Takes average latitude and longitude calculated in beahc_w_to_gps.ipynb and computes bearing using bearing formula
    \(\theta =\>\mathrm{atan2}\>(\sin \Delta \lambda \cdot \cos \phi _{2},\cos \phi _{1}\cdot \sin \phi _{2}-\sin \phi _{1}\cdot \cos \phi _{2}\cdot \cos \Delta \lambda )\).'''
    
    lat = np.radians(ds['mean_lat'])
    lon = np.radians(ds['mean_lon'])

    lat1 = lat.shift(site=1)
    lon1 = lon.shift(site=1)
    lat2 = lat.shift(site=-1)
    lon2 = lon.shift(site=-1)

    dlon = lon2 - lon1
    
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)

    bearing = np.degrees(np.arctan2(y, x))
    bearing = (bearing + 360) % 360
    bearing_clean = bearing.fillna(0)

    return ds.assign(bearing=bearing_clean)

In [27]:
def calc_qs_mu(dp, hs, tp, angles, k, d=8): # mean direction of propagation bearing from north (degrees), significant wave height (meters), period (seconds), k, shoreface depth (meters) 

    '''Calculates Q_s and diffusivity (mu) using formula from Ashton, Murray 2006 B, equation (7) and (8). Negative sign is added so that positive Q_s 
    corresponds to rightward sediment movement when facing offshore (in this case, roughly south-bound).'''
    
    tp_da = tp['tp'] if isinstance(tp, xr.Dataset) else tp # just in case
    hs_da = hs['hs'] if isinstance(hs, xr.Dataset) else hs
    dp_da = dp['dp'] if isinstance(dp, xr.Dataset) else dp

    phi = dp_da - 90
    thet = ((angles + 90) % 360) - 90 # assumes no shoreline bearing angles equal to 270
    angle_term = ((phi - thet + 180) % 360) - 180
    angle_term = angle_term.where(np.abs(angle_term) <= 90) # mask areas where waves appear to be moving away from shore
    angle_term_abs = np.abs(angle_term)
    rad_factor = np.pi / 180
    angle_term = angle_term * rad_factor
    angle_term_abs = angle_term_abs * rad_factor
    cos_term = np.cos(angle_term)
    sin_term = np.sin(angle_term)
    sin_term_abs = np.sin(angle_term_abs)
    cos_term_abs = np.cos(angle_term_abs)
    power_term = ((tp_da * cos_term) ** 0.2) * (hs_da ** 2.4) 
    power_term_abs = ((tp_da * cos_term_abs) ** 0.2) * (hs_da ** 2.4)
    coeff = k / d
    mu = -(coeff * power_term_abs * ((1.2 * (sin_term_abs**2)) - (cos_term_abs**2)))
    qs = -k * (tp_da ** .2) * (hs_da ** 2.4) * (cos_term ** 1.2) * sin_term # CERC formula, negative fixes the bearing assumption so that downdrift qs is positive
    # qs = -k * (hs_da ** 2) * cos_term * sin_term # breaking wave formula
    return qs, mu, angle_term

In [6]:
def get_u_a(ds, start_date, end_date):

    '''Calculates U, the fraction of high angle wave contribution to Q_s over the total wave contribution to Q_s over the given time period.
    High-angle threshold is defined by the absolute angle between wave crests and the shoreline being greater than pi/4.'''

    '''Calculates A, the fraction of high rightward wave contribution to Q_s over the total wave contribution to Q_s over the given time period.'''
    
    ds_subset = ds.sel(time=slice(start_date, end_date))
    
    high_mask = (ds_subset['angle_diff'] > (np.pi/4)) | (ds_subset['angle_diff'] < -(np.pi/4))
    low_mask  = (ds_subset['angle_diff'] <= (np.pi/4)) & (ds_subset['angle_diff'] >= -(np.pi/4))
    qs_high = np.abs(ds_subset['qs'].where(high_mask)).sum(dim='time')
    qs_low  = np.abs(ds_subset['qs'].where(low_mask)).sum(dim='time')
    total1 = qs_low + qs_high
    u = (qs_high / total1)

    left_mask = (ds_subset['angle_diff'] < 0)
    right_mask = (ds_subset['angle_diff'] >= 0)
    qs_left = np.abs(ds_subset['qs'].where(left_mask)).sum(dim='time')
    qs_right  = np.abs(ds_subset['qs'].where(right_mask)).sum(dim='time')
    total2 = qs_left + qs_right
    a = (qs_left / total2)
    
    return u, a

In [7]:
dp = xr.open_dataset('dp_merged_all.nc') # peak direction
hs = xr.open_dataset('hs_merged_all.nc') # wave height
tp = xr.open_dataset('tp_merged_all.nc') # peak period
shore = xr.open_dataset('NC_average_lat_lon.nc') # shoreline points

In [8]:
# compute shoreline angles (bearing from north)
angles_xr = get_angles_xr(shore)

In [28]:
# iterate through wave sample ids, time periods, and latitude boundaries and calculate sediment transport statistics
id_list = [2844, 3096, 3142]

time_list = [
    ['1984-01-01T12:00:00', '2004-01-01T12:00:00'],
    ['2004-01-01T12:00:00', '2024-01-01T12:00:00']
]

lat_bounds = [
    [35.237, 36.642]
]

# compute mean shoreline angle between two latitude bounds
bearing_list = []

for i in range(len(lat_bounds)):
    angles = angles_xr.where((angles_xr['mean_lat'] >= lat_bounds[i][0]) & (angles_xr['mean_lat'] <= lat_bounds[i][1]), drop=True) 
    
    rad = np.radians(angles['bearing'])

    mean_sin = np.sin(rad).mean(dim='site')
    mean_cos = np.cos(rad).mean(dim='site')

    mean_angle_rad = np.arctan2(mean_sin, mean_cos)
    mean_bearing = np.degrees(mean_angle_rad)
    mean_bearing = (mean_bearing + 360) % 360

    bearing_list.append(mean_bearing.values)

k = calc_alongshore_transport_k()

results = []

# calculate statistics
for i in range(len(id_list)):
    site_id = id_list[i]
    
    dp_site = dp.sel(site=site_id)
    hs_site = hs.sel(site=site_id)
    tp_site = tp.sel(site=site_id)
    
    for j in range(len(bearing_list)):
        bearing_val = bearing_list[j]

        qs_da, mu_da, angle_term_da = calc_qs_mu(
            dp=dp_site, hs=hs_site, tp=tp_site, angles=bearing_val, k=k
        )
        
        site_wave_ds = xr.Dataset({
            'qs': qs_da,
            'angle_diff': angle_term_da, 
            'mu': mu_da
        })

        lat_window = lat_bounds[j]
        
        for m in range(len(time_list)):
            start = time_list[m][0]
            end = time_list[m][1]

            u_val, a_val = get_u_a(site_wave_ds, start_date=start, end_date=end)

            time_subset = site_wave_ds.sel(time=slice(start, end))

            mu_avg = time_subset['mu'].mean(dim='time')
            qs_sum = time_subset['qs'].sum(dim='time')

            results.append({
                'site': site_id,
                'lat_bounds': lat_window,
                'time_window': f"{start} to {end}",
                'bearing': bearing_val,
                'mu_avg': float(mu_avg.values),
                'qs_sum': float(qs_sum.values),
                'u': float(u_val.values),
                'a': float(a_val.values)
            })

df_results = pd.DataFrame(results)
print(df_results)            

   site        lat_bounds                                 time_window  \
0  2844  [35.237, 36.642]  1984-01-01T12:00:00 to 2004-01-01T12:00:00   
1  2844  [35.237, 36.642]  2004-01-01T12:00:00 to 2024-01-01T12:00:00   
2  3096  [35.237, 36.642]  1984-01-01T12:00:00 to 2004-01-01T12:00:00   
3  3096  [35.237, 36.642]  2004-01-01T12:00:00 to 2024-01-01T12:00:00   
4  3142  [35.237, 36.642]  1984-01-01T12:00:00 to 2004-01-01T12:00:00   
5  3142  [35.237, 36.642]  2004-01-01T12:00:00 to 2024-01-01T12:00:00   

              bearing    mu_avg       qs_sum         u         a  
0  348.32407775843853 -0.000200  6651.077148  0.455431  0.736683  
1  348.32407775843853  0.002097  7176.770996  0.444971  0.739102  
2  348.32407775843853  0.001867  6997.814453  0.447642  0.736374  
3  348.32407775843853  0.004049  7826.160645  0.433944  0.746694  
4  348.32407775843853  0.004504  3945.253906  0.421048  0.639709  
5  348.32407775843853  0.005376  4676.594727  0.422716  0.653259  


In [29]:
# iterate through wave sample ids, time periods, and latitude boundaries and calculate sediment transport statistics
id_list = [2844, 3096, 3142]

time_list = [
    ['1992-01-01T12:00:00', '2007-01-01T12:00:00'],
    ['2008-01-01T12:00:00', '2024-01-01T12:00:00']
]

lat_bounds = [
    [35.231204, 35.774414]
]

# compute mean shoreline angle between two latitude bounds
bearing_list = []

for i in range(len(lat_bounds)):
    angles = angles_xr.where((angles_xr['mean_lat'] >= lat_bounds[i][0]) & (angles_xr['mean_lat'] <= lat_bounds[i][1]), drop=True) 
    
    rad = np.radians(angles['bearing'])

    mean_sin = np.sin(rad).mean(dim='site')
    mean_cos = np.cos(rad).mean(dim='site')

    mean_angle_rad = np.arctan2(mean_sin, mean_cos)
    mean_bearing = np.degrees(mean_angle_rad)
    mean_bearing = (mean_bearing + 360) % 360

    bearing_list.append(mean_bearing.values)

k = calc_alongshore_transport_k()

results = []

# calculate statistics
for i in range(len(id_list)):
    site_id = id_list[i]
    
    dp_site = dp.sel(site=site_id)
    hs_site = hs.sel(site=site_id)
    tp_site = tp.sel(site=site_id)
    
    for j in range(len(bearing_list)):
        bearing_val = bearing_list[j]

        qs_da, mu_da, angle_term_da = calc_qs_mu(
            dp=dp_site, hs=hs_site, tp=tp_site, angles=bearing_val, k=k
        )
        
        site_wave_ds = xr.Dataset({
            'qs': qs_da,
            'angle_diff': angle_term_da, 
            'mu': mu_da
        })

        lat_window = lat_bounds[j]
        
        for m in range(len(time_list)):
            start = time_list[m][0]
            end = time_list[m][1]

            u_val, a_val = get_u_a(site_wave_ds, start_date=start, end_date=end)

            time_subset = site_wave_ds.sel(time=slice(start, end))

            mu_avg = time_subset['mu'].mean(dim='time')
            qs_sum = time_subset['qs'].sum(dim='time')

            results.append({
                'site': site_id,
                'lat_bounds': lat_window,
                'time_window': f"{start} to {end}",
                'bearing': bearing_val,
                'mu_avg': float(mu_avg.values),
                'qs_sum': float(qs_sum.values),
                'u': float(u_val.values),
                'a': float(a_val.values)
            })

df_results = pd.DataFrame(results)
print(df_results)            

   site              lat_bounds                                 time_window  \
0  2844  [35.231204, 35.774414]  1992-01-01T12:00:00 to 2007-01-01T12:00:00   
1  2844  [35.231204, 35.774414]  2008-01-01T12:00:00 to 2024-01-01T12:00:00   
2  3096  [35.231204, 35.774414]  1992-01-01T12:00:00 to 2007-01-01T12:00:00   
3  3096  [35.231204, 35.774414]  2008-01-01T12:00:00 to 2024-01-01T12:00:00   
4  3142  [35.231204, 35.774414]  1992-01-01T12:00:00 to 2007-01-01T12:00:00   
5  3142  [35.231204, 35.774414]  2008-01-01T12:00:00 to 2024-01-01T12:00:00   

             bearing    mu_avg       qs_sum         u         a  
0  4.774102075972678 -0.007095  4500.614746  0.462131  0.689162  
1  4.774102075972678 -0.007132  4837.582031  0.471966  0.681926  
2  4.774102075972678 -0.006186  5248.870605  0.454671  0.708885  
3  4.774102075972678 -0.006920  5729.078125  0.463174  0.703914  
4  4.774102075972678 -0.001715  3337.818848  0.430248  0.626926  
5  4.774102075972678 -0.003004  3792.326172  0.447